<a href="https://colab.research.google.com/github/Dayaanaly/U-Net-vs-SegNet/blob/main/Demo_Unet_vs_SegNet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## **Demo de segmentación con modelo entrenado**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

MessageError: Error: credential propagation was unsuccessful

1. Cargar ambos modelos

In [ ]:
# ================================
# Cargar modelos entrenados automáticamente
# ================================

output_dir = '/content/drive/MyDrive/Equipo Trifuerza/Modelos entrenados/Comparativa_Unet_SegNet_80e'

# Buscar modelos guardados de U-Net y SegNet
unet_candidates = glob.glob(os.path.join(output_dir, '*UNet*.keras'))
segnet_candidates = glob.glob(os.path.join(output_dir, '*SegNet*.keras'))

if len(unet_candidates) == 0:
    raise FileNotFoundError("No se encontró ningún modelo U-Net .keras en la carpeta indicada.")

if len(segnet_candidates) == 0:
    raise FileNotFoundError("No se encontró ningún modelo SegNet .keras en la carpeta indicada.")

# Tomar el modelo más reciente de cada arquitectura
unet_model_path = max(unet_candidates, key=os.path.getmtime)
segnet_model_path = max(segnet_candidates, key=os.path.getmtime)

print("Modelo U-Net seleccionado:")
print(unet_model_path)

print("\nModelo SegNet seleccionado:")
print(segnet_model_path)

# Cargar U-Net
model_unet = load_model(
    unet_model_path,
    custom_objects=custom_objects_dict,
    compile=False
)

# Cargar SegNet
model_segnet = load_model(
    segnet_model_path,
    custom_objects=custom_objects_dict,
    compile=False
)

print("\nModelos cargados correctamente.")

NameError: name 'glob' is not defined

2. Hacer inferencia con ambos modelos

In [ ]:
demo_image_batch = np.expand_dims(demo_image, axis=0)

In [ ]:
# ================================
# Inferencia con U-Net y SegNet
# ================================

# Predicción con U-Net
pred_unet = model_unet.predict(demo_image_batch)
pred_unet_class = np.argmax(pred_unet, axis=-1)[0]

# Predicción con SegNet
pred_segnet = model_segnet.predict(demo_image_batch)
pred_segnet_class = np.argmax(pred_segnet, axis=-1)[0]

print("Forma de salida U-Net:", pred_unet.shape)
print("Forma de salida SegNet:", pred_segnet.shape)

print("Clases detectadas por U-Net:", np.unique(pred_unet_class))
print("Clases detectadas por SegNet:", np.unique(pred_segnet_class))

3. Demo visual con ambas arquitecturas

In [ ]:
# ================================
# Demo con máscara real, U-Net y SegNet
# ================================

mask_dir = '/content/drive/MyDrive/Equipo Trifuerza/Datos/Data set grises/Mascaras mezcladas'

mask_files = sorted([
    f for f in os.listdir(mask_dir)
    if f.lower().endswith(valid_extensions)
])

mask_path = os.path.join(mask_dir, mask_files[idx])

print("Máscara real seleccionada:")
print(mask_path)

# Cargar máscara real
true_mask = load_image_custom(
    mask_path,
    target_size=(IMG_HEIGHT, IMG_WIDTH),
    color_mode='grayscale'
)

true_mask = true_mask.astype(np.int32)

if true_mask.ndim == 3 and true_mask.shape[-1] == 1:
    true_mask = np.squeeze(true_mask, axis=-1)

# Aplicar las mismas correcciones usadas durante el entrenamiento
true_mask = np.rot90(true_mask, k=3, axes=(0, 1))

true_mask = cv2.resize(
    true_mask,
    (IMG_WIDTH, IMG_HEIGHT),
    interpolation=cv2.INTER_NEAREST
)

true_mask = np.fliplr(true_mask)
true_mask = np.clip(true_mask, 0, NUM_CLASSES - 1)

# ================================
# Visualización completa
# ================================

plt.figure(figsize=(18, 5))

plt.subplot(1, 4, 1)
plt.imshow(demo_image.squeeze(), cmap='gray')
plt.title('Imagen de entrada')
plt.axis('off')

plt.subplot(1, 4, 2)
plt.imshow(
    true_mask,
    cmap='nipy_spectral',
    vmin=0,
    vmax=NUM_CLASSES - 1
)
plt.title('Máscara real')
plt.axis('off')

plt.subplot(1, 4, 3)
plt.imshow(
    pred_unet_class,
    cmap='nipy_spectral',
    vmin=0,
    vmax=NUM_CLASSES - 1
)
plt.title('Predicción U-Net')
plt.axis('off')

plt.subplot(1, 4, 4)
plt.imshow(
    pred_segnet_class,
    cmap='nipy_spectral',
    vmin=0,
    vmax=NUM_CLASSES - 1
)
plt.title('Predicción SegNet')
plt.axis('off')

plt.tight_layout()
plt.show()